# LangGraph Streaming

## What is Streaming?

**Streaming** allows you to receive graph outputs incrementally as nodes execute, rather than waiting for the entire graph to complete. This is essential for:

- **Real-time UI updates** (chatbots, progress indicators)

- **Long-running workflows** (see intermediate results)

- **Debugging** (monitor execution step-by-step)

- **Better UX** (users see progress immediately)

## How to Stream

Use `graph.stream()` instead of `graph.invoke()`:

In [ ]:
# ❌ No streaming - wait for everything
result = graph.invoke(input_state, config)

# ✅ Streaming - get updates as they happen
for chunk in graph.stream(input_state, config):
    print(chunk)

---

## Stream Modes

LangGraph supports **5 different stream modes** to control what data you receive:

| **Mode** | **Returns** | **Use Case** |

|----------|-------------|--------------|

| `values` | Full state after each node | See complete state evolution |

| `updates` | Only what each node added/changed | See node contributions |

| `messages` | New messages from LLMs | Chat applications |

| `debug` | Detailed execution info | Debugging and monitoring |

| `custom` | User-defined streaming | Advanced custom logic |

---

## 1. Stream Mode: `values`

Returns the **complete state** after each node executes.

In [ ]:
for chunk in graph.stream(input_state, stream_mode="values"):
    print(chunk)

**Output**: Full state dict at each step

**Best for**: Seeing how state evolves over time

---

## 2. Stream Mode: `updates`

Returns **only the updates** (changes) each node made to the state.

In [ ]:
for chunk in graph.stream(input_state, stream_mode="updates"):
    print(chunk)

**Output**: `{node_name: {field: new_value}}`

**Best for**: Understanding what each node contributes

---

## 3. Stream Mode: `messages`

Returns **only new messages** added by LLM nodes (for chat applications).

In [ ]:
for chunk in graph.stream(input_state, stream_mode="messages"):
    print(chunk)

**Output**: New AIMessage or HumanMessage objects

**Best for**: Chatbots with streaming LLM responses

---

## 4. Stream Mode: `debug`

Returns **detailed debugging information** including metadata, timing, and execution details.

In [ ]:
for chunk in graph.stream(input_state, stream_mode="debug"):
    print(chunk)

**Output**: Rich debug info with timestamps, node names, task IDs

**Best for**: Monitoring, logging, debugging

---

## 5. Stream Mode: `custom`

Use **custom streaming** with `StreamWriter` for advanced use cases.

In [ ]:
from langgraph.types import StreamWriter

def custom_node(state: State, writer: StreamWriter):
    writer("Custom event 1")
    # ... do work ...
    writer("Custom event 2")
    return state

for chunk in graph.stream(input_state, stream_mode="custom"):
    print(chunk)

**Best for**: Custom progress indicators, metrics, events

---

## Subgraph Streaming

Enable `subgraphs=True` to stream from nested subgraphs:

In [ ]:
for chunk in graph.stream(
    input_state,
    stream_mode="updates",
    subgraphs=True  # Include subgraph outputs
):
    print(chunk)

**Output format**: `((subgraph_path,), {node_name: updates})`

- `()` = parent graph

- `('subgraph_name:id',)` = subgraph level

---

## Multiple Stream Modes

Stream multiple modes simultaneously:

In [ ]:
for chunk in graph.stream(
    input_state,
    stream_mode=["values", "updates"]
):
    mode, data = chunk
    if mode == "values":
        print(f"State: {data}")
    elif mode == "updates":
        print(f"Update: {data}")

---

## Async Streaming

For async applications, use `astream()`:

In [ ]:
async for chunk in graph.astream(input_state, stream_mode="updates"):
    print(chunk)
    await process_chunk(chunk)

---

## Complete Streaming Example

In [ ]:
from langgraph.graph import StateGraph, START, END
from typing_extensions import TypedDict

class State(TypedDict):
    count: int
    messages: list[str]

def step1(state: State):
    return {"count": state["count"] + 1, "messages": state["messages"] + ["Step 1"]}

def step2(state: State):
    return {"count": state["count"] * 2, "messages": state["messages"] + ["Step 2"]}

# Build graph
workflow = StateGraph(State)
workflow.add_node("step1", step1)
workflow.add_node("step2", step2)
workflow.add_edge(START, "step1")
workflow.add_edge("step1", "step2")
workflow.add_edge("step2", END)
graph = workflow.compile()

# Stream with different modes
print("=== STREAM MODE: values ===")
for chunk in graph.stream({"count": 0, "messages": []}, stream_mode="values"):
    print(chunk)

print("\n=== STREAM MODE: updates ===")
for chunk in graph.stream({"count": 0, "messages": []}, stream_mode="updates"):
    print(chunk)

In [ ]:
**Output (values mode)**:
{'count': 1, 'messages': ['Step 1']}
{'count': 2, 'messages': ['Step 1', 'Step 2']}

In [ ]:
**Output (updates mode)**:
{'step1': {'count': 1, 'messages': ['Step 1']}}
{'step2': {'count': 2, 'messages': ['Step 2']}}

---

## Real-World Use Cases

In [ ]:
### 1. Chatbot with Streaming Response
for chunk in graph.stream(user_input, stream_mode="messages"):
    # Stream each token to UI
    display_message_chunk(chunk)

In [ ]:
### 2. Progress Bar
total_nodes = 5
for i, chunk in enumerate(graph.stream(input, stream_mode="updates")):
    progress = (i + 1) / total_nodes * 100
    update_progress_bar(progress)

In [ ]:
### 3. Real-time Monitoring
for chunk in graph.stream(input, stream_mode="debug"):
    log_to_monitoring_system(chunk)
    check_for_errors(chunk)

---

## Key Points

1. ✅ **stream()** returns generator - iterate to get chunks

2. ✅ **stream_mode** controls what data you receive

3. ✅ **subgraphs=True** streams from nested graphs

4. ✅ **astream()** for async applications

5. ✅ **Multiple modes** can be used together

6. ✅ **Checkpointing works** - each streamed step is saved

---

## When to Use Each Mode

| **Scenario** | **Recommended Mode** |

|--------------|---------------------|

| Chat application | `messages` |

| Progress tracking | `updates` |

| State visualization | `values` |

| Debugging | `debug` |

| Custom events | `custom` |

| General purpose | `updates` or `values` |

In [ ]:
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI

load_dotenv()

api_key = os.environ['UNIFIED_LLM_KEY']
# print(api_key)
base_url = ""

llm = ChatOpenAI(
    model="gpt-4o-mini",
    temperature=0,
    max_tokens=256,
    api_key=api_key,
    base_url=base_url
)

In [ ]:
from langgraph.graph import START, StateGraph
from typing import TypedDict

# Define subgraph
class SubgraphState(TypedDict):
    foo: str  # note that this key is shared with the parent graph state
    bar: str

def subgraph_node_1(state: SubgraphState):
    return {"bar": "bar"}

def subgraph_node_2(state: SubgraphState):
    return {"foo": state["foo"] + state["bar"]}

subgraph_builder = StateGraph(SubgraphState)
subgraph_builder.add_node(subgraph_node_1)
subgraph_builder.add_node(subgraph_node_2)
subgraph_builder.add_edge(START, "subgraph_node_1")
subgraph_builder.add_edge("subgraph_node_1", "subgraph_node_2")
subgraph = subgraph_builder.compile()

# Define parent graph
class ParentState(TypedDict):
    foo: str

def node_1(state: ParentState):
    return {"foo": "hi! " + state["foo"]}

builder = StateGraph(ParentState)
builder.add_node("node_1", node_1)
builder.add_node("node_2", subgraph)
builder.add_edge(START, "node_1")
builder.add_edge("node_1", "node_2")
graph = builder.compile()

for chunk in graph.stream(
    {"foo": "foo"},
    stream_mode="updates",
    # Set subgraphs=True to stream outputs from subgraphs
    subgraphs=True,
):
    print(chunk)

In [ ]:
for chunk in graph.stream(
    {"foo": "foo"},
    stream_mode="values",
    # Set subgraphs=True to stream outputs from subgraphs
    subgraphs=True,
):
    print(chunk)

In [ ]:
for chunk in graph.stream(
    {"foo": "foo"},
    stream_mode="messages",
    # Set subgraphs=True to stream outputs from subgraphs
    subgraphs=True,
):
    print(chunk)

---

## Practical Example: Stream Mode "messages" with LLM

Let's create a chatbot that streams LLM responses in real-time, similar to ChatGPT.

In [ ]:
from typing_extensions import TypedDict, Annotated
from langgraph.graph import StateGraph, START, END
from langgraph.graph.message import add_messages

# Define state for chat
class ChatbotState(TypedDict):
    messages: Annotated[list, add_messages]
    status: str

# Create chatbot node using the LLM
def chatbot_node(state: ChatbotState):
    """Call LLM with conversation history"""
    response = llm.invoke(state["messages"])
    return {"messages": [response]}

def start_node(state: ChatbotState):
    return {"status": "INITIATED"}

def end_node(state: ChatbotState):
    return {"status": "FINISHED"}

# Build chatbot graph
chatbot_builder = StateGraph(ChatbotState)
chatbot_builder.add_node("chatbot", chatbot_node)
chatbot_builder.add_node("start_node", start_node)
chatbot_builder.add_node("end_node", end_node)
chatbot_builder.add_edge(START, "start_node")
chatbot_builder.add_edge("start_node", "chatbot")
chatbot_builder.add_edge("chatbot", "end_node")
chatbot_builder.add_edge("end_node", END)

chatbot_graph = chatbot_builder.compile()

print("✅ Chatbot graph created with LLM")

### Test 1: Stream Mode "updates" - See what the chatbot node returns

In [ ]:
print("=" * 80)
print("STREAM MODE: updates")
print("=" * 80)

for chunk in chatbot_graph.stream(
    {"messages": [("user", "What is LangGraph in one sentence?")]},
    stream_mode="updates"
):
    print(chunk)
    print()

### Test 2: Stream Mode "messages" - Get ONLY the AI message (streaming response)

In [ ]:
print("=" * 80)
print("STREAM MODE: messages (streaming AI response)")
print("=" * 80)

for chunk in chatbot_graph.stream(
    {"messages": [("user", "Explain streaming in LangGraph in 2 sentences.")]},
    stream_mode="messages"
):
    # Each chunk is a new message (typically the AI response)
    print(f"Message chunk: {chunk}")
    print(f"Type: {type(chunk)}")
    print(f"Content: {chunk.content if hasattr(chunk, 'content') else chunk}")
    print("-" * 40)

### Test 3: Stream Mode "values" - See complete state evolution

In [ ]:
print("=" * 80)
print("STREAM MODE: values")
print("=" * 80)

for i, chunk in enumerate(chatbot_graph.stream(
    {"messages": [("user", "What are the benefits of streaming?")]},
    stream_mode="values"
)):
    print(f"\nStep {i + 1}:")
    print(f"  Message count: {len(chunk['messages'])}")
    for j, msg in enumerate(chunk['messages']):
        role = "User" if hasattr(msg, 'type') and msg.type == "human" else "AI"
        content = msg.content if hasattr(msg, 'content') else str(msg)
        print(f"  [{role}]: {content[:100]}...")  # Truncate for readability

### Test 4: Multi-turn Conversation with Streaming

Demonstrate a real conversation with history using stream mode "messages".

In [ ]:
print("=" * 80)
print("MULTI-TURN CONVERSATION with stream_mode='messages'")
print("=" * 80)

# Start conversation
conversation_state = {"messages": []}

# Turn 1
print("\n👤 User: Hello! What is LangGraph?")
for msg_chunk in chatbot_graph.stream(
    {"messages": [("user", "Hello! What is LangGraph?")]},
    stream_mode="messages"
):
    conversation_state["messages"].append(msg_chunk)
    print(f"🤖 AI: {msg_chunk}")

# Turn 2 - Continue with history
print("\n👤 User: Can you give me an example?")
for msg_chunk in chatbot_graph.stream(
    {
        "messages": conversation_state["messages"] +
        [("user", "Can you give me an example?")]
    },
    stream_mode="messages"
):
    conversation_state["messages"].append(msg_chunk)
    print(f"🤖 AI: {msg_chunk}")

print("\n" + "=" * 80)
print(f"Total messages in conversation: {len(conversation_state['messages'])}")

### Summary: Stream Mode Comparison

| **Stream Mode** | **What You Get** | **When to Use** |

|-----------------|------------------|-----------------|

| `updates` | Node-level updates: `{'chatbot': {'messages': [AIMessage(...)]}}` | Debug individual node outputs |

| `messages` | Only message objects: `AIMessage(content="...")` | **Chatbots - streaming responses** |

| `values` | Full state at each step: `{'messages': [HumanMessage(...), AIMessage(...)]}` | See conversation history evolution |

**For chatbots**: Use `stream_mode="messages"` to get clean, streamable AI responses without wrapper objects!